In [ ]:
!nvidia-smi -l 1

In [ ]:
%pip install transformer_lens
%pip install circuitsvis

In [ ]:
# import ipykernel
# import subprocess
# import platform

# def spawn_matlab_style_console():
#     conn_file = ipykernel.connect.get_connection_file()
#     cmd = f"jupyter console --existing {conn_file}"
#     sys_os = platform.system()
    
#     try:
#         if sys_os == 'Windows':
#             # Opens a new Command Prompt
#             subprocess.Popen(f'start cmd /k {cmd}', shell=True)
#         elif sys_os == 'Darwin':
#             # Opens a new macOS Terminal
#             subprocess.Popen(['osascript', '-e', f'tell application "Terminal" to do script "{cmd}"'])
#         elif sys_os == 'Linux':
#             # Attempts to open default Linux terminal
#             subprocess.Popen(['x-terminal-emulator', '-e', cmd])
#         print("✅ MATLAB-style console launched in a new window!")
#     except Exception as e:
#         print(f"Failed to launch terminal: {e}")

# spawn_matlab_style_console()

In [ ]:
import ipykernel
from IPython.display import display, HTML

# Get the exact absolute path of the current active kernel
conn_file = ipykernel.connect.get_connection_file()
cmd = f"jupyter console --existing {conn_file}"

# Generate a UI button to copy the command
html = f"""
<div style="background: #1e1e1e; color: #d4d4d4; padding: 12px; border-radius: 4px; font-family: monospace; border: 1px solid #333;">
    <span id="jupyter_cmd">{cmd}</span><br><br>
    <button onclick="navigator.clipboard.writeText(document.getElementById('jupyter_cmd').innerText);" 
            style="background: #007acc; color: white; border: none; padding: 6px 12px; border-radius: 3px; cursor: pointer;">
        Copy to VSCode Terminal
    </button>
</div>
"""
display(HTML(html))

In [ ]:
import sys
sys.path.append('/home/galk/LanguageDynamics/src') 

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm import tqdm

import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.distributions.kl import kl_divergence
from torch.distributions.categorical import Categorical
from torch.distributions.multivariate_normal import MultivariateNormal
# from datasets import load_dataset
from datetime import datetime

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# import interpretability stuff
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import HookedTransformer, FactoredMatrix
import circuitsvis as cv

from importlib import reload

models_path = '/home/galk/LanguageDynamics/models/linguistic_flip_flop'

In [ ]:
## Activation Patching Functions

from functools import partial

def patch_head_hook(corrupted_hook, hook, clean_cache, head_idx, position=None):
    """
    The hook function that surgically overwrites the corrupted activations 
    with the clean activations for a specific head and position.
    
    corrupted_hook shape: [batch, sequence_length, n_heads, d_head]
    """
    # Fetch the clean activations from the cache
    clean_hook = clean_cache[hook.name]
    
    if position is None:
        # Wide Patching: Patch this head at ALL sequence positions
        corrupted_hook[:, :, head_idx, :] = clean_hook[:, :, head_idx, :]
    else:
        # Position-Specific Patching: Patch this head at a SINGLE position
        corrupted_hook[:, position, head_idx, :] = clean_hook[:, position, head_idx, :]
        
    return corrupted_hook


def run_attention_patching(
    model: HookedTransformer,
    clean_prompt: str,
    corrupt_prompt: str,
    clean_answer: str,
    corrupt_answer: str,
    layer: int,
    head_idx: int,
    position: int = None
):
    """
    Runs the full causal tracing experiment and returns logits and the recovery score.
    """
    # 1. Ensure prompts are tokenized to the same length for clean position mapping
    clean_tokens = model.to_tokens(clean_prompt)
    corrupt_tokens = model.to_tokens(corrupt_prompt)
    
    if clean_tokens.shape[1] != corrupt_tokens.shape[1]:
        print("Warning: Prompts have different token lengths. Patching by position may behave unexpectedly.")

    # Get token IDs for the expected answers to calculate our metric
    clean_answer_id = model.to_single_token(clean_answer)
    corrupt_answer_id = model.to_single_token(corrupt_answer)

    # 2. Run the CLEAN prompt and cache all activations
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)

    # 3. Run the CORRUPT prompt to get baseline corrupted logits
    corrupt_logits = model(corrupt_tokens)

    # 4. Run the PATCHED prompt
    # We hook into the 'z' activation (the mixed values right before the W_O projection)
    hook_name = f"blocks.{layer}.attn.hook_z"
    
    # Use functools.partial to freeze our specific arguments into the hook function
    hook_fn = partial(
        patch_head_hook, 
        clean_cache=clean_cache, 
        head_idx=head_idx, 
        position=position
    )
    
    patched_logits = model.run_with_hooks(
        corrupt_tokens,
        fwd_hooks=[(hook_name, hook_fn)]
    )

    # 5. Calculate Metrics (Logit Difference at the final token position)
    def get_logit_diff(logits):
        # Look at the final token's prediction [batch_idx=0, pos_idx=-1]
        final_logits = logits[0, -1, :]
        return (final_logits[clean_answer_id] - final_logits[corrupt_answer_id]).item()

    clean_diff = get_logit_diff(clean_logits)
    corrupt_diff = get_logit_diff(corrupt_logits)
    patched_diff = get_logit_diff(patched_logits)

    # Recovery Score: 0% means it acts like the corrupted model, 100% means it acts like the clean model
    recovery_score = (patched_diff - corrupt_diff) / (clean_diff - corrupt_diff)

    return {
        "clean_logits": clean_logits,
        "corrupt_logits": corrupt_logits,
        "patched_logits": patched_logits,
        "clean_diff": clean_diff,
        "corrupt_diff": corrupt_diff,
        "patched_diff": patched_diff,
        "recovery_score": recovery_score
    }

In [ ]:
# Load model
device = utils.get_device()
model_name = "attn-only-2l"
model = HookedTransformer.from_pretrained(model_name, device=device)

In [ ]:
# generate
prompt = "I always thought that one day I would make it to the moon. I mean, why wouldn't I?\nI'm not sure that one day I would make it to the moon.\nI'm not sure that one day I would make it to the moon.\nSo many millions of tokens of Arbitrary Noise that interrupt the prompt repeatedly that seem unrelated.\nWill this matter to the model at all?\nI believe that it should, because the doctor said so."
noise = "So many millions of tokens of Arbitrary Noise that interrupt the prompt repeatedly that seem unrelated.\nWill this matter to the model at all?\nI believe that it should, because the doctor said so."
input_tokens = model.to_tokens(prompt)

generated = model.generate(
    prompt,
    max_new_tokens=100,
    do_sample=False,
    temperature=1.0,
    top_p=1.0,
    prepend_bos=True
)

print(generated)

In [ ]:
# Run with cache
prompt = " I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q"
input_tokens = model.to_tokens(prompt)
# input_tokens = model.to_tokens(generated)
# print(input_tokens.device)
logits, cache = model.run_with_cache(input_tokens, remove_batch_dim=True)

In [ ]:
# Get attention patterns and visualize them
layer = 1
print(type(cache))
attention_pattern = cache["attn", layer]
print(attention_pattern.shape) # [head_idx, destination, source]
str_tokens = model.to_str_tokens(prompt)

In [ ]:
print(f"Layer {layer} head Attention Patterns:")
cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern)

In [ ]:
## Plot Attention Coeffs
head = 6
jump = 5
offset = 80
plt.plot(attention_pattern[head, offset].cpu().numpy(), '--o')
# plt.plot(attention_pattern[head, offset].cpu().numpy()[offset-jump+1::-jump][::-1], '--o')
plt.ylim(bottom=0)
plt.xlabel("Position")
plt.ylabel("Attention Weight")
plt.grid()
# plt.show()

# print(attention_pattern[head, -1].cpu().numpy().sum())

## Plot Token Relative Attention Coeff Vs. Repeats
relative_coeffs = [attention_pattern[head, pos].cpu().numpy()[pos-jump+1::-jump].sum() / attention_pattern[head, pos].cpu().numpy().sum() for pos in np.arange(jump+np.mod(offset, jump), len(attention_pattern[head, offset].cpu().numpy()), jump)]
# print(attention_pattern[head, offset].cpu().numpy()[offset-jump+1::-jump].sum() / attention_pattern[head, offset].cpu().numpy()[0])
plt.figure()
plt.plot(relative_coeffs)
plt.xlabel("#Repeats")
plt.ylabel("Relative Coeff")
plt.ylim(bottom=0, top=1)
plt.grid()
plt.show()

In [ ]:
layer = 1

# 1. The Residual Stream
# shape: [batch, sequence_length, d_model]
resid_pre = cache[f"blocks.{layer}.hook_resid_pre"] # Before attention
resid_post = cache[f"blocks.{layer}.hook_resid_post"] # After attention & addition

# 2. The Q, K, and V Projections
# shape: [batch, sequence_length, num_heads, d_head]
q_vectors = cache[f"blocks.{layer}.attn.hook_q"] # shape [position, head_idx, d_head]
k_vectors = cache[f"blocks.{layer}.attn.hook_k"]
v_vectors = cache[f"blocks.{layer}.attn.hook_v"]
z_vectors = cache[f"blocks.{layer}.attn.hook_z"]

# 3. Attention Scores
# hook_attn_scores: Unnormalized dot products (Q*K^T)
# hook_pattern: Normalized probabilities (after Softmax)
# shape: [batch, num_heads, query_pos, key_pos]
unnormalized_scores = cache[f"blocks.{layer}.attn.hook_attn_scores"]
attention_pattern = cache[f"blocks.{layer}.attn.hook_pattern"]

# 4. Computed Outputs (Before being added to residual stream)
# hook_z: The output of the OV circuit before the final W_O projection
# hook_result: The final output of each head after W_O projection (the exact H_t we modeled)
# shape (hook_result): [batch, sequence_length, num_heads, d_model]
# head_outputs = cache[f"blocks.{layer}.attn.hook_result"]

# 5. Final Logits
# Already returned from run_with_cache. shape: [batch, sequence_length, vocab_size]
print(f"Logits shape: {logits.shape}")

In [ ]:
## Plot Z norms Vs. Repeats

head = 6
jump = 5
z_norms = np.linalg.norm(z_vectors[:,head].cpu().numpy(), axis=-1)
z_norms = z_norms[1:] # remove the <BOS> token
# z_norms = z_norms[2:] # remove prefix

plt.figure(figsize=(10, 8))
plt.plot(np.reshape(z_norms, (-1, jump)))
plt.grid()
plt.xlabel("#Repeats")
plt.ylabel("Z norm")
plt.ylim(bottom=0)
plt.show()

In [ ]:
## Plot V-vectors and Z-vectors Cosine Similarity Vs. Position

v_cosine_sim = F.cosine_similarity(v_vectors[-2, head], z_vectors[:,head], dim=-1)
plt.figure(figsize=(10, 8))
plt.plot(v_cosine_sim.cpu().numpy())
# plt.plot(z_norms[offset::jump])
plt.grid()
plt.xlabel("Position")
plt.ylabel("Cosine Sim")
plt.ylim(bottom=0, top=1)
plt.show()

In [ ]:
## Causal Patching Experiments

# Setup our experiment variables
clean_prompt =   " I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M Q I R P M"
clean_answer = " Q" # Notice the leading space! Tokenization matters.

corrupt_prompt = " I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M A I R P M"
corrupt_answer = " A"

target_layer = 1
target_head = 7 # Let's say we suspect L1H4 is an induction head
target_position = -1 # The position of the second name token

# --- Experiment 1: Wide Patching (All positions) ---
print("\n--- Running Wide Patching ---")
results_wide = run_attention_patching(
    model, clean_prompt, corrupt_prompt, clean_answer, corrupt_answer,
    layer=target_layer, head_idx=target_head, position=None
)
print(f"Clean Logit Diff:   {results_wide['clean_diff']:.4f}")
print(f"Corrupt Logit Diff: {results_wide['corrupt_diff']:.4f}")
print(f"Patched Logit Diff: {results_wide['patched_diff']:.4f}")
print(f"Recovery Score:     {results_wide['recovery_score']:.2%}")

# # --- Experiment 2: Position-Specific Patching ---
# print(f"\n--- Running Position-Specific Patching (Pos: {target_position}) ---")
# results_pos = run_attention_patching(
#     model, clean_prompt, corrupt_prompt, clean_answer, corrupt_answer,
#     layer=target_layer, head_idx=target_head, position=target_position
# )
# print(f"Patched Logit Diff: {results_pos['patched_diff']:.4f}")
# print(f"Recovery Score:     {results_pos['recovery_score']:.2%}")

In [ ]:
from tqdm.auto import tqdm

def run_scaling_patching_experiment(
    model: HookedTransformer, 
    base_seq: str, 
    clean_target: str, 
    corrupt_target: str, 
    max_repetitions: int = 10,
    prefix: str|None = None
):
    """
    Runs causal tracing across all layers and heads for increasing repetitions.
    Returns a numpy array of shape [n_layers, n_heads, max_repetitions]
    """
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    
    # Initialize the results array. 
    # Shape: [layer, head_idx, n_repetitions]
    # Note: Index 0 on the last dimension corresponds to N=1 repetition.
    recovery_scores = np.zeros((n_layers, n_heads, max_repetitions))
    
    clean_answer_id = model.to_single_token(clean_target)
    corrupt_answer_id = model.to_single_token(corrupt_target)

    min_repetitions = 1 if prefix is None else 0

    # Loop over the number of repetitions
    for n_rep in tqdm(range(min_repetitions, max_repetitions + 1), desc="Repetitions"):
        
        # 1. Dynamically construct the prompts for this N
        if prefix is not None:
            clean_prompt = prefix + (base_seq + clean_target) * n_rep + base_seq
            corrupt_prompt = prefix + (base_seq + corrupt_target) * n_rep + base_seq
        else:
            clean_prompt = (base_seq + clean_target) * n_rep + base_seq
            corrupt_prompt = (base_seq + corrupt_target) * n_rep + base_seq
        
        clean_tokens = model.to_tokens(clean_prompt)
        corrupt_tokens = model.to_tokens(corrupt_prompt)
        
        # 2. Run clean and corrupt baseline ONCE per N to save compute
        clean_logits, clean_cache = model.run_with_cache(clean_tokens)
        corrupt_logits = model(corrupt_tokens)
        
        def get_logit_diff(logits):
            final_logits = logits[0, -1, :]
            return (final_logits[clean_answer_id] - final_logits[corrupt_answer_id]).item()
            
        clean_diff = get_logit_diff(clean_logits)
        corrupt_diff = get_logit_diff(corrupt_logits)
        
        # 3. Iterate over all layers and heads
        for layer in range(n_layers):
            hook_name = f"blocks.{layer}.attn.hook_z"

            for head_idx in range(n_heads):    
                
                # We use wide patching (position=None) across the whole sequence
                hook_fn = partial(
                    patch_head_hook, 
                    clean_cache=clean_cache, 
                    head_idx=head_idx, 
                    position=None 
                )
                
                patched_logits = model.run_with_hooks(
                    corrupt_tokens,
                    fwd_hooks=[(hook_name, hook_fn)]
                )
                
                patched_diff = get_logit_diff(patched_logits)
                
                # Calculate and store the recovery score
                # Handle edge case where clean_diff == corrupt_diff (e.g. at N=0 if model knows nothing)
                if clean_diff == corrupt_diff:
                    score = 0.0
                else:
                    score = (patched_diff - corrupt_diff) / (clean_diff - corrupt_diff)
                    
                recovery_scores[layer, head_idx, n_rep - 1] = score

    return recovery_scores

# Define the building blocks of your sequence
base_sequence = " seems right. It"
clean_target = " just"
corrupt_target = " like"
prefix = "I always thought that I would make it to the moon. Why wouldn't I? It just"

max_reps = 100 #

print("Running experiment...")
results_matrix = run_scaling_patching_experiment( # shape [layer, head, n_repetition]
    model, 
    base_seq=base_sequence, 
    clean_target=clean_target, 
    corrupt_target=corrupt_target,
    max_repetitions=max_reps,
    prefix=prefix
)

print(f"Finished! Matrix shape: {results_matrix.shape}")

In [ ]:
# Plot reocvery scores vs. repetition
for layer in [1, 0]:
    plt.figure(figsize=(10, 8))
    for i_head in range(model.cfg.n_heads):
        plt.plot(results_matrix[layer][i_head], label=f"L{layer}H{i_head}")

    plt.xlabel("# Repetitions")
    plt.ylabel("Recovery Score [%]")
    plt.title("Recovery Score vs. # Repetitions")
    plt.legend()
    plt.grid()
plt.show()

In [ ]:
layer = 1
plt.figure(figsize=(10, 8))

plt.plot(results_matrix[layer][6] + results_matrix[layer][7], label="L1H6 + L1H7")

plt.xlabel("# Repetitions")
plt.ylabel("Recovery Score [%]")
plt.title("Recovery Score vs. # Repetitions")
plt.legend()
plt.grid()

In [ ]:
# Compute autocorrelation matrix of v_vectors for the selected head
v_head = v_vectors[:, head]  # shape: [sequence_length, d_head]

# Compute pairwise cosine similarity (autocorrelation)
autocorr_matrix = F.cosine_similarity(v_head.unsqueeze(1), v_head.unsqueeze(0), dim=-1)

# Plot as heatmap
plt.figure(figsize=(12, 10))
plt.imshow(autocorr_matrix.cpu().numpy(), cmap='coolwarm', aspect='auto')
plt.colorbar(label='Cosine Similarity')
plt.xlabel('Position')
plt.ylabel('Position')
plt.title(f'Autocorrelation Matrix of V Vectors (head {head})')
plt.tight_layout()
plt.show()